<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/torch/basic/custom-loss/custom-loss.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("custom-loss", ...)`


# Problem: Implement Custom Loss Function (Huber Loss)

### Problem Statement
You are tasked with implementing the **Huber Loss** as a custom loss function in PyTorch. The Huber loss is a robust loss function used in regression tasks, less sensitive to outliers than Mean Squared Error (MSE). It transitions between L2 loss (squared error) and L1 loss (absolute error) based on a threshold parameter $ \delta $.

The Huber loss is mathematically defined as:
$$
L_{\delta}(y, \hat{y}) = 
\begin{cases} 
\frac{1}{2}(y - \hat{y})^2 & \text{for } |y - \hat{y}| \leq \delta, \\
\delta \cdot (|y - \hat{y}| - \frac{1}{2} \delta) & \text{for } |y - \hat{y}| > \delta,
\end{cases}
$$

where:
- $y$ is the true value,
- $\hat{y}$ is the predicted value,
- $\delta$ is a threshold parameter that controls the transition between L1 and L2 loss.

### Requirements
1. **Custom Loss Function**:
   - Implement a class `HuberLoss(delta=1.0)` inheriting from `torch.nn.Module`.
   - Define `forward(y_pred, y_true)` to compute the Huber loss elementwise as per the formula and return its **mean** over all elements (a scalar) — the same as `nn.HuberLoss(delta=delta)` with its default `reduction='mean'`.

2. **Usage in a Regression Model**:
   - Integrate the custom loss function into a regression training pipeline.
   - Use it to compute and optimize the loss during model training.

### Constraints
- The implementation must handle both scalar and batch inputs for $ y $ (true values) and $ \hat{y} $ (predicted values).
- Do not call `nn.HuberLoss` / `F.huber_loss` / `F.smooth_l1_loss` in your implementation (you can use them to check your answer).

Note: the two pieces meet with matching value and slope at $|y - \hat{y}| = \delta$, so the loss is continuously differentiable, and its gradient is bounded by $\delta$ — which is why a few outliers cannot dominate training the way they do with MSE. PyTorch's `SmoothL1Loss(beta)` is the same curve divided by `beta`, so the two coincide only when $\delta = \beta = 1$.


Extra Details: https://en.wikipedia.org/wiki/Huber_loss

<details>
  <summary>💡 Hint</summary>
  Some details: https://www.kaggle.com/code/bigironsphere/loss-function-library-keras-pytorch/notebook
</details>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Generate synthetic data
torch.manual_seed(42)
X = torch.rand(100, 1) * 10  # 100 data points between 0 and 10
y = 2 * X + 3 + torch.randn(100, 1)  # Linear relationship with noise

# TODO: Define the nn.Module for the Huber Loss
class HuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        ...

    def forward(self, y_pred, y_true):
        # TODO: elementwise Huber loss, then the mean over all elements
        ...


# Define the Linear Regression Model
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)  # Single input and single output

    def forward(self, x):
        return self.linear(x)

# Initialize the model, loss function, and optimizer
model = LinearRegressionModel()
# TODO: Use your HuberLoss (delta=1.0) as the criterion
criterion = ...
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    predictions = model(X)
    loss = criterion(predictions, y)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Log progress every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

In [ ]:
# Display the learned parameters
[w, b] = model.linear.parameters()
print(f"Learned weight: {w.item():.4f}, Learned bias: {b.item():.4f}")

# Testing on new data
X_test = torch.tensor([[4.0], [7.0]])
with torch.no_grad():
    predictions = model(X_test)
    print(f"Predictions for {X_test.tolist()}: {predictions.tolist()}")

In [ ]:
# Why Huber? Corrupt every 10th target with a large outlier and fit the same model
# with MSE and with Huber. MSE's gradient grows with the error, so the outliers drag
# the line up; Huber's gradient is capped at delta, so the fit stays near y = 2x + 3.
y_outliers = y.clone()
y_outliers[::10] += 30

def fit(criterion, epochs=3000):
    torch.manual_seed(0)
    m = LinearRegressionModel()
    opt = optim.SGD(m.parameters(), lr=0.01)
    for _ in range(epochs):
        opt.zero_grad()
        criterion(m(X), y_outliers).backward()
        opt.step()
    return m.linear.weight.item(), m.linear.bias.item()

print("True line:  w = 2.000, b = 3.000")
print("MSE fit:    w = {:.3f}, b = {:.3f}".format(*fit(nn.MSELoss())))
print("Huber fit:  w = {:.3f}, b = {:.3f}".format(*fit(HuberLoss(delta=1.0))))